# Protein Structure Analysis

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Describe the four levels of protein structure (primary through quaternary)
- Parse PDB files and understand ATOM record format
- Use BioPython's Bio.PDB module to navigate the Structure/Model/Chain/Residue/Atom hierarchy
- Calculate interatomic distances, bond angles, and RMSD
- Assign and analyze secondary structure with DSSP
- Perform structural alignment and superposition (Kabsch algorithm)
- Understand molecular surfaces (van der Waals, solvent-accessible, electrostatic)
- Generate and interpret Ramachandran plots
- Use visualization tools: PyMOL commands and nglview in Jupyter
- Appreciate modern structure prediction with AlphaFold

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Phylogenetics: Reconstructing Evolutionary History](../../Tier_2_Core_Bioinformatics/06_Phylogenetics/01_phylogenetics.ipynb) | [Next: Nucleic Acid Structure →](../../Tier_2_Core_Bioinformatics/08_Nucleic_Acid_Structure/01_nucleic_acid_structure.ipynb)

---

---

## 1. The Four Levels of Protein Structure

Proteins are organized in a structural hierarchy:

```
PRIMARY          SECONDARY           TERTIARY            QUATERNARY

M-V-L-S-P-A     alpha-helix          Folded              Multiple
  (sequence)     beta-sheet           single chain        subunits
                 loops/turns

  MVLSPAD...       /\/\               .---.              .---. .---.
                  / helix\            /     \            /  A  X  B  \
                 ====sheet           | domain|          |     | |     |
                                     \     /            \  C  X  D  /
                                      '---'              '---' '---'

 Amino acid       Local H-bond       Complete 3D         Assembly of
 sequence         patterns           fold of one         multiple
                                     polypeptide         polypeptides
```

### Primary Structure
The linear sequence of amino acids, read from N-terminus to C-terminus. Determined by the gene encoding the protein. All higher-order structure ultimately derives from this sequence.

### Secondary Structure
Local, regular folding patterns stabilized by backbone hydrogen bonds:

| Element | H-bond Pattern | Residues/Turn | Rise/Residue | Phi/Psi Angles |
|---------|---------------|---------------|--------------|----------------|
| Alpha-helix | i -> i+4 | 3.6 | 1.5 A | -57, -47 |
| 3_10-helix | i -> i+3 | 3.0 | 2.0 A | -49, -26 |
| Pi-helix | i -> i+5 | 4.4 | 1.15 A | -57, -70 |
| Beta-sheet (parallel) | between strands | - | 3.2 A | -119, +113 |
| Beta-sheet (antiparallel) | between strands | - | 3.4 A | -139, +135 |

### Tertiary Structure
The complete three-dimensional arrangement of all atoms in a single polypeptide chain, stabilized by:
- Hydrophobic interactions (dominant driving force)
- Hydrogen bonds
- Salt bridges (ionic interactions)
- Disulfide bonds (covalent, Cys-Cys)
- Van der Waals forces

### Quaternary Structure
The arrangement of multiple polypeptide subunits into a functional complex. Examples:
- **Hemoglobin**: alpha2-beta2 tetramer
- **DNA polymerase III**: ~10 different subunits
- **Ribosome**: dozens of protein + RNA subunits

---

## 2. The PDB File Format

The Protein Data Bank (PDB) format is the standard for representing 3D macromolecular structures.

### File Sections

```
HEADER     Title, classification, date, PDB ID
TITLE      Descriptive title of the structure
COMPND     Compound (protein) information
SOURCE     Organism source
EXPDTA     Experimental method (X-RAY, NMR, CRYO-EM)
REMARK     Comments and experimental details
DBREF      Database cross-references (UniProt, etc.)
SEQRES     Primary sequence
HELIX      Alpha-helix assignments
SHEET      Beta-sheet assignments
CRYST1     Unit cell parameters
ATOM       Standard residue atomic coordinates  <-- Main data
HETATM     Non-standard residues (ligands, water)
CONECT     Connectivity for ligands
END        End of file
```

### ATOM Record Format (Fixed-Width Columns)

```
ATOM      1  N   ALA A   1      11.104   6.134  -6.504  1.00 41.88           N
|         |  |   |   |   |      |        |       |      |    |              |
|         |  |   |   |   |      |        |       |      |    |              Element
|         |  |   |   |   |      |        |       |      |    B-factor (A^2)
|         |  |   |   |   |      |        |       |      Occupancy
|         |  |   |   |   |      |        |       Z coordinate (A)
|         |  |   |   |   |      |        Y coordinate (A)
|         |  |   |   |   |      X coordinate (A)
|         |  |   |   |   Residue sequence number
|         |  |   |   Chain ID
|         |  |   Residue name (3-letter code)
|         |  Atom name (N, CA, C, O, CB, ...)
|         Atom serial number
Record type

Column positions (1-indexed):
 1-6    Record type ("ATOM  " or "HETATM")
 7-11   Atom serial number
13-16   Atom name
17      Alternate location indicator
18-20   Residue name
22      Chain ID
23-26   Residue sequence number
31-38   X coordinate (Angstroms)
39-46   Y coordinate
47-54   Z coordinate
55-60   Occupancy (0.0-1.0)
61-66   B-factor / temperature factor (A^2)
77-78   Element symbol
```

**B-factor** (temperature factor): indicates atomic mobility/disorder. High B-factors suggest flexible regions; low B-factors suggest well-ordered regions.

**Occupancy**: fraction of time the atom occupies this position (relevant for alternate conformations).

In [ ]:
def parse_atom_line(line):
    """
    Parse a single ATOM/HETATM line from a PDB file.
    PDB format uses fixed-width columns, NOT whitespace-delimited.
    """
    return {
        'record_type': line[0:6].strip(),
        'serial': int(line[6:11]),
        'atom_name': line[12:16].strip(),
        'alt_loc': line[16].strip(),
        'res_name': line[17:20].strip(),
        'chain_id': line[21],
        'res_seq': int(line[22:26]),
        'x': float(line[30:38]),
        'y': float(line[38:46]),
        'z': float(line[46:54]),
        'occupancy': float(line[54:60]),
        'b_factor': float(line[60:66]),
        'element': line[76:78].strip() if len(line) > 76 else ''
    }


# Demonstrate with a sample ATOM line
example_line = "ATOM      1  N   ALA A   1      11.104   6.134  -6.504  1.00 41.88           N"
atom = parse_atom_line(example_line)

print("Parsed ATOM record:")
for key, value in atom.items():
    print(f"  {key:12s}: {value}")

In [ ]:
def parse_pdb_atoms(pdb_lines):
    """
    Parse all ATOM and HETATM records from PDB content.
    
    Parameters:
        pdb_lines: list of strings (lines from a PDB file)
    Returns:
        dict with 'atoms', 'hetatoms', 'chains', 'header' keys
    """
    atoms = []
    hetatoms = []
    chains = set()
    header = {}

    for line in pdb_lines:
        record = line[0:6]

        if record == 'HEADER':
            header['classification'] = line[10:50].strip()
            header['pdb_id'] = line[62:66].strip()

        elif record == 'EXPDTA':
            header['method'] = line[10:80].strip()

        elif record == 'ATOM  ':
            atom = parse_atom_line(line)
            atoms.append(atom)
            chains.add(atom['chain_id'])

        elif record == 'HETATM':
            hetatoms.append(parse_atom_line(line))

    return {
        'atoms': atoms,
        'hetatoms': hetatoms,
        'chains': sorted(chains),
        'header': header
    }


# Demonstrate with sample PDB content
sample_pdb = [
    "HEADER    PLANT PROTEIN                           22-JAN-81   1CRN              ",
    "EXPDTA    X-RAY DIFFRACTION                                                     ",
    "ATOM      1  N   THR A   1      17.047  14.099   3.625  1.00 13.79           N  ",
    "ATOM      2  CA  THR A   1      16.967  12.784   4.338  1.00 10.80           C  ",
    "ATOM      3  C   THR A   1      15.685  12.755   5.133  1.00  9.19           C  ",
    "ATOM      4  O   THR A   1      15.268  13.825   5.594  1.00  9.85           O  ",
    "ATOM      5  CB  THR A   1      18.170  12.703   5.337  1.00 13.02           C  ",
    "ATOM      6  N   THR A   2      15.115  11.555   5.265  1.00  7.68           N  ",
    "ATOM      7  CA  THR A   2      13.856  11.469   6.066  1.00  8.43           C  ",
    "ATOM      8  C   THR A   2      14.164  10.785   7.379  1.00  5.00           C  ",
    "ATOM      9  O   THR A   2      14.993   9.862   7.425  1.00  6.07           O  ",
    "HETATM  328  O   HOH A 101      -1.545  10.029   5.904  1.00 26.30           O  ",
]

result = parse_pdb_atoms(sample_pdb)
print(f"PDB ID: {result['header'].get('pdb_id', 'N/A')}")
print(f"Method: {result['header'].get('method', 'N/A')}")
print(f"Chains: {result['chains']}")
print(f"ATOM records: {len(result['atoms'])}")
print(f"HETATM records: {len(result['hetatoms'])}")

---

## 3. Parsing PDB with BioPython (Bio.PDB)

BioPython provides a powerful, object-oriented PDB parser built around the **SMCRA hierarchy**:

```
Structure
  |-- Model (0, 1, ...)
        |-- Chain ('A', 'B', ...)
              |-- Residue (('', 10, ''), 'ALA')
                    |-- Atom ('CA', 'N', 'C', 'O', ...)

S = Structure   One PDB entry (may have multiple models for NMR)
M = Model       One set of coordinates (X-ray: 1 model; NMR: ~20 models)
C = Chain       One polypeptide or nucleic acid chain
R = Residue     One amino acid or nucleotide
A = Atom        One atom with (x, y, z) coordinates
```

Each level is iterable, so you can use nested loops or list comprehensions to traverse the hierarchy.

In [ ]:
from Bio.PDB import PDBParser, PDBList
import warnings
warnings.filterwarnings('ignore')  # Suppress PDB parsing warnings for cleaner output

# Download a structure from RCSB PDB
# Using crambin (1CRN) -- a small, well-resolved plant protein (46 residues)
pdbl = PDBList()
pdb_file = pdbl.retrieve_pdb_file('1CRN', pdir='pdb_files', file_format='pdb')
print(f"Downloaded: {pdb_file}")

In [ ]:
# Parse the downloaded structure
parser = PDBParser(QUIET=True)
structure = parser.get_structure('crambin', pdb_file)

# Navigate the SMCRA hierarchy
print("=== Structure Hierarchy ===")
print(f"Structure ID: {structure.id}")

for model in structure:
    print(f"\n  Model {model.id}")
    for chain in model:
        residues = list(chain.get_residues())
        # Filter standard amino acids (exclude water, ligands)
        aa_residues = [r for r in residues if r.id[0] == ' ']
        het_residues = [r for r in residues if r.id[0] != ' ']
        print(f"    Chain {chain.id}: {len(aa_residues)} amino acids, {len(het_residues)} het groups")
        
        # Show first 5 residues
        print(f"    First 5 residues:")
        for res in aa_residues[:5]:
            atoms = list(res.get_atoms())
            print(f"      {res.get_resname()} {res.id[1]}: {len(atoms)} atoms")

In [ ]:
# Accessing individual atoms and their properties
model = structure[0]
chain = model['A']

# Get a specific residue (residue 10 in chain A)
# Residue ID is a tuple: (hetflag, resseq, insertion_code)
residue = chain[(' ', 10, ' ')]
# Shorthand (works when hetflag=' ' and icode=' '):
residue = chain[10]

print(f"Residue: {residue.get_resname()} {residue.id[1]}")
print(f"\nAtoms in this residue:")
for atom in residue:
    coord = atom.get_vector()
    print(f"  {atom.get_name():4s}  element={atom.element:2s}  "
          f"coords=({coord[0]:7.3f}, {coord[1]:7.3f}, {coord[2]:7.3f})  "
          f"B-factor={atom.get_bfactor():.2f}")

In [ ]:
# Extract all CA atoms (backbone trace)
import numpy as np

ca_atoms = []
for residue in chain:
    if residue.id[0] != ' ':  # Skip heteroatoms
        continue
    if 'CA' in residue:
        ca = residue['CA']
        ca_atoms.append({
            'res_name': residue.get_resname(),
            'res_num': residue.id[1],
            'coord': ca.get_vector().get_array()
        })

print(f"Extracted {len(ca_atoms)} CA atoms")
print(f"\nSequence from CA trace:")
# Three-letter to one-letter conversion
AA_MAP = {
    'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
    'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
    'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
    'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
}
sequence = ''.join(AA_MAP.get(ca['res_name'], 'X') for ca in ca_atoms)
print(sequence)

---

## 4. Calculating Distances, Angles, and RMSD

Structural analysis frequently requires measuring geometric properties:

- **Distance**: Euclidean distance between two atoms
- **Angle**: Angle formed by three atoms
- **Dihedral angle**: Torsion angle defined by four atoms
- **RMSD**: Root Mean Square Deviation between two coordinate sets

```
RMSD = sqrt( (1/N) * sum( |r_i - r_i'|^2 ) )

Interpretation:
  0 - 1 A    Nearly identical structures
  1 - 2 A    Very similar (same fold)
  2 - 3 A    Similar fold, some variation
  3 - 5 A    Same topology, different details
  > 5 A      Different structures
```

In [ ]:
import numpy as np

def distance(coord1, coord2):
    """Euclidean distance between two 3D points."""
    return np.linalg.norm(np.array(coord1) - np.array(coord2))


def angle(coord1, coord2, coord3):
    """Angle (in degrees) at coord2 formed by coord1-coord2-coord3."""
    v1 = np.array(coord1) - np.array(coord2)
    v2 = np.array(coord3) - np.array(coord2)
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def dihedral(p1, p2, p3, p4):
    """Dihedral (torsion) angle in degrees defined by four points."""
    p1, p2, p3, p4 = [np.array(p) for p in (p1, p2, p3, p4)]
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    n1 /= np.linalg.norm(n1)
    n2 /= np.linalg.norm(n2)
    m1 = np.cross(n1, b2 / np.linalg.norm(b2))
    return np.degrees(np.arctan2(np.dot(m1, n2), np.dot(n1, n2)))


def calculate_rmsd(coords1, coords2):
    """RMSD between two Nx3 coordinate arrays."""
    coords1, coords2 = np.array(coords1), np.array(coords2)
    if coords1.shape != coords2.shape:
        raise ValueError(f"Shape mismatch: {coords1.shape} vs {coords2.shape}")
    diff = coords1 - coords2
    return np.sqrt(np.mean(np.sum(diff**2, axis=1)))


# Demonstrate with crambin residues
model = structure[0]
chain = model['A']

# Distance between CA atoms of residues 1 and 2
ca1 = chain[1]['CA'].get_vector().get_array()
ca2 = chain[2]['CA'].get_vector().get_array()
print(f"CA-CA distance (res 1--2): {distance(ca1, ca2):.2f} A")

# N-CA-C bond angle for residue 5
res5 = chain[5]
n_coord = res5['N'].get_vector().get_array()
ca_coord = res5['CA'].get_vector().get_array()
c_coord = res5['C'].get_vector().get_array()
print(f"N-CA-C angle (res 5):     {angle(n_coord, ca_coord, c_coord):.1f} degrees")

# Peptide bond dihedral (omega) between residues 5 and 6
res6 = chain[6]
omega = dihedral(
    res5['CA'].get_vector().get_array(),
    res5['C'].get_vector().get_array(),
    res6['N'].get_vector().get_array(),
    res6['CA'].get_vector().get_array()
)
print(f"Omega angle (res 5--6):   {omega:.1f} degrees (expected ~180 for trans)")

In [ ]:
# BioPython also provides built-in distance calculation via the minus operator
atom1 = chain[1]['CA']
atom2 = chain[2]['CA']

# The - operator on Atom objects returns distance in Angstroms
d = atom1 - atom2
print(f"BioPython atom distance: {d:.2f} A")

# Contact map: find all CA pairs within 8 Angstroms
ca_list = []
for residue in chain:
    if residue.id[0] == ' ' and 'CA' in residue:
        ca_list.append(residue['CA'])

contacts = []
for i in range(len(ca_list)):
    for j in range(i + 4, len(ca_list)):  # Skip nearby in sequence
        d = ca_list[i] - ca_list[j]
        if d < 8.0:
            contacts.append((i + 1, j + 1, d))

print(f"\nCA-CA contacts (< 8 A, |i-j| >= 4): {len(contacts)}")
print("First 10 contacts:")
for res_i, res_j, dist in contacts[:10]:
    print(f"  Res {res_i:3d} -- Res {res_j:3d}: {dist:.2f} A")

---

## 5. Secondary Structure Assignment with DSSP

DSSP (Define Secondary Structure of Proteins) is the gold-standard algorithm for assigning secondary structure from 3D coordinates. It uses hydrogen bond energies calculated from N-H...O=C geometry.

### DSSP Codes

| Code | Structure | Description |
|------|-----------|-------------|
| H | Alpha-helix | 4-turn helix (i -> i+4 H-bonds) |
| B | Beta-bridge | Isolated beta-bridge |
| E | Beta-strand | Extended strand in beta-sheet |
| G | 3_10-helix | 3-turn helix (i -> i+3) |
| I | Pi-helix | 5-turn helix (i -> i+5) |
| T | Turn | H-bonded turn |
| S | Bend | High curvature region |
| - | Coil | No regular structure |

BioPython wraps the DSSP program (must be installed separately via `conda install -c salilab dssp` or `apt install dssp`).

In [ ]:
from Bio.PDB.DSSP import DSSP

# Run DSSP on our structure
# Requires the 'dssp' or 'mkdssp' executable to be installed
try:
    dssp = DSSP(structure[0], pdb_file, dssp='mkdssp')
    
    print(f"DSSP assigned {len(dssp)} residues\n")
    
    # DSSP returns: (dssp_index, amino_acid, ss, acc, phi, psi, ...)
    ss_sequence = ''
    for key in dssp.keys():
        residue_dssp = dssp[key]
        ss = residue_dssp[2]  # Secondary structure code
        ss_sequence += ss
    
    print(f"Secondary structure assignment:")
    print(f"  {ss_sequence}")
    
    # Count secondary structure composition
    total = len(ss_sequence)
    helix = ss_sequence.count('H') + ss_sequence.count('G') + ss_sequence.count('I')
    sheet = ss_sequence.count('E') + ss_sequence.count('B')
    coil = total - helix - sheet
    
    print(f"\nSecondary structure composition:")
    print(f"  Helix (H+G+I): {helix:3d} ({100*helix/total:.1f}%)")
    print(f"  Sheet (E+B):   {sheet:3d} ({100*sheet/total:.1f}%)")
    print(f"  Other:         {coil:3d} ({100*coil/total:.1f}%)")

except Exception as e:
    print(f"DSSP not available: {e}")
    print("Install with: conda install -c salilab dssp")
    print("\nFalling back to PDB HELIX/SHEET records...")
    print("(PDB files contain pre-calculated secondary structure in HELIX and SHEET records)")

In [ ]:
# Extract per-residue DSSP data including solvent accessibility
try:
    dssp_data = []
    for key in dssp.keys():
        chain_id, res_id = key
        residue_dssp = dssp[key]
        dssp_data.append({
            'chain': chain_id,
            'res_num': res_id[1],
            'aa': residue_dssp[1],
            'ss': residue_dssp[2],
            'acc': residue_dssp[3],   # Solvent accessibility (A^2)
            'phi': residue_dssp[4],
            'psi': residue_dssp[5],
        })
    
    # Display first 15 residues
    print(f"{'Res':>4s} {'AA':>3s} {'SS':>3s} {'ACC':>6s} {'Phi':>8s} {'Psi':>8s}")
    print("-" * 36)
    for d in dssp_data[:15]:
        print(f"{d['res_num']:4d} {d['aa']:>3s} {d['ss']:>3s} {d['acc']:6.1f} {d['phi']:8.1f} {d['psi']:8.1f}")

except NameError:
    print("DSSP data not available (see cell above).")

---

## 6. Ramachandran Plot

The Ramachandran plot displays the backbone dihedral angles phi and psi for each residue. It reveals which regions of conformational space are sterically allowed.

```
                         psi (degrees)
                          +180
                           |
              beta-sheet   |   polyproline II
               region      |    region
       +---------+----+----+----+--------+
       |         |XXXX|    |    |        |
       |         |XXXX|    |    |        |
 -180 -+---------+----+----+----+--------+- +180  phi
       |         |    |    |    |        |
       |         |    |   XXXX  |        |
       |         |    |   XXXX  |        |  alpha-helix
       +---------+----+----+----+--------+   region
                           |
                          -180

  XXXX = Most populated regions (>99% of non-glycine residues)

  Typical angles:
    alpha-helix:  phi ~ -57, psi ~ -47
    beta-sheet:   phi ~ -120, psi ~ +120
    left-handed:  phi ~ +60,  psi ~ +60  (glycine only)
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def calculate_phi_psi_biopython(chain):
    """
    Calculate phi/psi angles for all residues in a chain using BioPython.
    
    phi = dihedral(C_prev, N, CA, C)
    psi = dihedral(N, CA, C, N_next)
    """
    from Bio.PDB.Polypeptide import PPBuilder
    ppb = PPBuilder()
    
    phi_psi_list = []
    for pp in ppb.build_peptides(chain):
        phi_psi = pp.get_phi_psi_list()
        for i, (phi, psi) in enumerate(phi_psi):
            if phi is not None and psi is not None:
                phi_psi_list.append({
                    'residue': pp[i].get_resname(),
                    'res_num': pp[i].id[1],
                    'phi': np.degrees(phi),
                    'psi': np.degrees(psi)
                })
    return phi_psi_list


# Calculate phi/psi for crambin
phi_psi_data = calculate_phi_psi_biopython(structure[0]['A'])
print(f"Calculated phi/psi for {len(phi_psi_data)} residues")

# Display a few values
print(f"\n{'Res':>4s} {'#':>4s} {'Phi':>8s} {'Psi':>8s}")
print("-" * 28)
for d in phi_psi_data[:10]:
    print(f"{d['residue']:>4s} {d['res_num']:4d} {d['phi']:8.1f} {d['psi']:8.1f}")

In [ ]:
# Generate a Ramachandran plot
phi_values = [d['phi'] for d in phi_psi_data]
psi_values = [d['psi'] for d in phi_psi_data]
residues = [d['residue'] for d in phi_psi_data]

# Separate glycine (more conformational freedom)
gly_phi = [p for p, r in zip(phi_values, residues) if r == 'GLY']
gly_psi = [p for p, r in zip(psi_values, residues) if r == 'GLY']
other_phi = [p for p, r in zip(phi_values, residues) if r != 'GLY']
other_psi = [p for p, r in zip(psi_values, residues) if r != 'GLY']

fig, ax = plt.subplots(figsize=(8, 8))

# Plot allowed regions as background shading
# Alpha-helix region
from matplotlib.patches import Ellipse
alpha_region = Ellipse((-60, -45), 50, 50, alpha=0.1, color='blue', label='Alpha-helix region')
beta_region = Ellipse((-120, 130), 50, 50, alpha=0.1, color='green', label='Beta-sheet region')
ax.add_patch(alpha_region)
ax.add_patch(beta_region)

# Plot data points
ax.scatter(other_phi, other_psi, c='steelblue', s=30, alpha=0.7, label='Non-Gly residues')
ax.scatter(gly_phi, gly_psi, c='red', s=50, marker='^', alpha=0.8, label='Glycine')

ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xlabel('Phi (degrees)', fontsize=12)
ax.set_ylabel('Psi (degrees)', fontsize=12)
ax.set_title('Ramachandran Plot -- Crambin (1CRN)', fontsize=14)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

## 7. Structural Alignment and Superposition

Structural alignment answers the question: **how similar are two protein structures?**

Key differences from sequence alignment:

| Feature | Sequence Alignment | Structural Alignment |
|---------|-------------------|---------------------|
| Input | Amino acid letters | 3D coordinates |
| Method | Dynamic programming | Least-squares fitting |
| Score | % identity, E-value | RMSD (Angstroms), TM-score |
| Key insight | Sequences <20% identity can share the same fold |

### The Kabsch Algorithm

Finds the optimal rotation matrix to superimpose one structure onto another:

1. Center both structures at the origin
2. Compute the covariance matrix H = P^T * Q
3. Compute SVD: H = U * S * V^T
4. Rotation matrix: R = V * diag(1, 1, sign(det(V*U^T))) * U^T
5. Apply rotation, measure RMSD

In [ ]:
import numpy as np

def kabsch_superpose(mobile, reference):
    """
    Superimpose 'mobile' onto 'reference' using the Kabsch algorithm.
    
    Parameters:
        mobile:    Nx3 numpy array (structure to be moved)
        reference: Nx3 numpy array (fixed target)
    Returns:
        rotated:   Nx3 array (superimposed mobile coordinates)
        rmsd:      RMSD after optimal superposition
        R:         3x3 rotation matrix
    """
    mobile = np.array(mobile, dtype=float)
    reference = np.array(reference, dtype=float)
    
    # 1. Center both at origin
    mobile_center = mobile.mean(axis=0)
    ref_center = reference.mean(axis=0)
    mobile_c = mobile - mobile_center
    ref_c = reference - ref_center
    
    # 2. Covariance matrix
    H = mobile_c.T @ ref_c
    
    # 3. SVD
    U, S, Vt = np.linalg.svd(H)
    
    # 4. Correct for reflection
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T
    
    # 5. Apply rotation + translation
    rotated = (mobile_c @ R) + ref_center
    rmsd = calculate_rmsd(rotated, reference)
    
    return rotated, rmsd, R


# Demonstration: rotate crambin CA coordinates and recover them
ca_coords = np.array([ca['coord'] for ca in ca_atoms])

# Create a "rotated + translated" copy (simulating a second structure)
theta = np.radians(45)
R_test = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
])
mobile_test = (ca_coords @ R_test) + np.array([10, -5, 3])

print(f"RMSD before superposition: {calculate_rmsd(mobile_test, ca_coords):.3f} A")

aligned, rmsd, R_recovered = kabsch_superpose(mobile_test, ca_coords)
print(f"RMSD after superposition:  {rmsd:.6f} A")
print("(Near-zero RMSD confirms the algorithm correctly recovered the original orientation)")

In [ ]:
# BioPython provides a built-in Superimposer
from Bio.PDB import Superimposer

sup = Superimposer()

# Get CA atoms as Atom objects for two "structures"
# Here we use the same structure twice to demonstrate
fixed_atoms = []
moving_atoms = []
for residue in structure[0]['A']:
    if residue.id[0] == ' ' and 'CA' in residue:
        fixed_atoms.append(residue['CA'])
        moving_atoms.append(residue['CA'])  # Same atoms for demo

sup.set_atoms(fixed_atoms, moving_atoms)
print(f"BioPython Superimposer RMSD: {sup.rms:.4f} A")
print(f"Rotation matrix:\n{sup.rotran[0]}")
print(f"Translation vector: {sup.rotran[1]}")

### TM-score

RMSD is sensitive to outliers and depends on protein length. TM-score is a length-normalized measure that is more robust:

```
TM-score = (1/L_ref) * sum_i [ 1 / (1 + (d_i / d_0)^2) ]

where d_0 = 1.24 * (L_ref - 15)^(1/3) - 1.8

Interpretation:
  > 0.5    Same fold
  0.3-0.5  Possibly same fold
  < 0.3    Different folds
  ~ 1.0    Nearly identical
```

In [ ]:
def tm_score(coords1, coords2, L_ref=None):
    """
    Calculate TM-score between two aligned coordinate sets.
    
    Parameters:
        coords1, coords2: Nx3 arrays of aligned coordinates
        L_ref: Reference length (default: len(coords1))
    """
    coords1, coords2 = np.array(coords1), np.array(coords2)
    if L_ref is None:
        L_ref = len(coords1)
    
    d0 = 1.24 * (L_ref - 15) ** (1.0 / 3.0) - 1.8
    d0 = max(d0, 0.5)  # Minimum value
    
    distances = np.sqrt(np.sum((coords1 - coords2) ** 2, axis=1))
    scores = 1.0 / (1.0 + (distances / d0) ** 2)
    
    return np.sum(scores) / L_ref


# TM-score for our superposition
tm = tm_score(aligned, ca_coords)
print(f"TM-score: {tm:.4f}  (1.0 = identical)")

---

## 8. Molecular Surfaces

Molecular surfaces reveal how the protein interacts with its environment:

```
1. VAN DER WAALS SURFACE
   The union of atomic van der Waals spheres.
   Shows atomic radii. Has gaps between atoms.

2. SOLVENT-ACCESSIBLE SURFACE (SAS)
   Traced by the center of a rolling probe sphere (typically r=1.4 A for water).
   Represents the boundary accessible to solvent molecules.

3. SOLVENT-EXCLUDED SURFACE (SES) / Molecular Surface
   The contact surface + reentrant surface.
   Where solvent physically cannot penetrate.
   Smooth and continuous -- most common for visualization.
```

### Solvent-Accessible Surface Area (SASA)

SASA quantifies how much of each residue is exposed to solvent. This is critical for:
- Identifying **buried vs exposed** residues
- Estimating **binding interfaces** (buried SASA upon complex formation)
- Predicting **protein stability** (hydrophobic residues prefer to be buried)

In [ ]:
# Van der Waals radii for common elements (Angstroms)
VDW_RADII = {
    'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'H': 1.20,
    'P': 1.80, 'FE': 2.00, 'ZN': 1.39, 'MG': 1.73, 'SE': 1.90
}

# Maximum SASA for fully exposed amino acids (Angstroms^2)
MAX_SASA = {
    'ALA': 113, 'ARG': 241, 'ASN': 158, 'ASP': 151, 'CYS': 140,
    'GLN': 189, 'GLU': 183, 'GLY': 85,  'HIS': 194, 'ILE': 182,
    'LEU': 180, 'LYS': 211, 'MET': 204, 'PHE': 218, 'PRO': 143,
    'SER': 122, 'THR': 146, 'TRP': 259, 'TYR': 229, 'VAL': 160
}

# Kyte-Doolittle hydrophobicity scale
HYDROPHOBICITY = {
    'ALA': 1.8,  'ARG': -4.5, 'ASN': -3.5, 'ASP': -3.5, 'CYS': 2.5,
    'GLN': -3.5, 'GLU': -3.5, 'GLY': -0.4, 'HIS': -3.2, 'ILE': 4.5,
    'LEU': 3.8,  'LYS': -3.9, 'MET': 1.9,  'PHE': 2.8,  'PRO': -1.6,
    'SER': -0.8, 'THR': -0.7, 'TRP': -0.9, 'TYR': -1.3, 'VAL': 4.2
}


def classify_exposure(sasa, res_name):
    """Classify a residue as buried, intermediate, or exposed."""
    max_sasa = MAX_SASA.get(res_name, 150)
    relative = sasa / max_sasa
    if relative < 0.25:
        return 'buried'
    elif relative < 0.50:
        return 'intermediate'
    else:
        return 'exposed'


# SASA calculation using BioPython's ShrakeRupley
from Bio.PDB.SASA import ShrakeRupley

sr = ShrakeRupley()
sr.compute(structure[0], level='R')  # Compute per-residue SASA

print(f"{'Res':>4s} {'#':>4s} {'SASA':>7s} {'Max':>5s} {'Rel%':>5s} {'Exposure':>12s} {'Hydropathy':>10s}")
print("-" * 55)
for residue in structure[0]['A']:
    if residue.id[0] != ' ':
        continue
    name = residue.get_resname()
    num = residue.id[1]
    sasa = residue.sasa
    max_s = MAX_SASA.get(name, 150)
    rel = 100 * sasa / max_s
    exp = classify_exposure(sasa, name)
    hydro = HYDROPHOBICITY.get(name, 0)
    print(f"{name:>4s} {num:4d} {sasa:7.1f} {max_s:5d} {rel:5.1f} {exp:>12s} {hydro:10.1f}")
    if num >= 15:  # Show first 15 residues
        print("...")
        break

In [ ]:
# Visualize SASA and hydrophobicity profile
import matplotlib.pyplot as plt

sasa_values = []
hydro_values = []
res_labels = []

for residue in structure[0]['A']:
    if residue.id[0] != ' ':
        continue
    name = residue.get_resname()
    sasa_values.append(residue.sasa)
    hydro_values.append(HYDROPHOBICITY.get(name, 0))
    res_labels.append(f"{AA_MAP.get(name, 'X')}{residue.id[1]}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# SASA plot
ax1.bar(range(len(sasa_values)), sasa_values, color='steelblue', alpha=0.7)
ax1.set_ylabel('SASA (A^2)')
ax1.set_title('Solvent-Accessible Surface Area per Residue')
ax1.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Buried threshold')
ax1.legend()

# Hydrophobicity plot
colors = ['orangered' if h > 0 else 'dodgerblue' for h in hydro_values]
ax2.bar(range(len(hydro_values)), hydro_values, color=colors, alpha=0.7)
ax2.set_ylabel('Hydrophobicity')
ax2.set_xlabel('Residue')
ax2.set_title('Kyte-Doolittle Hydrophobicity')
ax2.axhline(y=0, color='black', linewidth=0.5)

# Set x-tick labels (every 5 residues)
tick_positions = range(0, len(res_labels), 5)
ax2.set_xticks(tick_positions)
ax2.set_xticklabels([res_labels[i] for i in tick_positions], rotation=45, ha='right')

plt.tight_layout()
plt.show()

### Electrostatic Surfaces

Electrostatic potential mapped onto the molecular surface reveals charge distribution:

- **Blue regions**: Positive charge (Arg, Lys, His) -- often DNA/RNA binding sites
- **Red regions**: Negative charge (Asp, Glu) -- often cation binding sites
- **White regions**: Neutral

Tools for electrostatic calculations:
- **APBS** (Adaptive Poisson-Boltzmann Solver): solves the Poisson-Boltzmann equation
- **PDB2PQR**: prepares PDB files for APBS by adding charges and radii
- **PyMOL APBS plugin**: integrates electrostatic surface calculation with visualization

---

## 9. Visualization Tools

### nglview (Jupyter-native 3D visualization)

`nglview` embeds an interactive 3D molecular viewer directly in Jupyter notebooks. Install with `pip install nglview`.

In [ ]:
# nglview: interactive 3D visualization in Jupyter
try:
    import nglview as nv
    
    # Load from BioPython structure
    view = nv.show_biopython(structure)
    
    # Apply cartoon representation colored by secondary structure
    view.clear_representations()
    view.add_cartoon(color='sstruc')  # Color by secondary structure
    view.add_ball_and_stick('hetero and not water')  # Show ligands
    
    # Set background
    view.background = 'white'
    
    print("Interactive viewer loaded. Drag to rotate, scroll to zoom.")
    display(view)

except ImportError:
    print("nglview is not installed. Install with:")
    print("  pip install nglview")
    print("  jupyter nbextension enable nglview --py --sys-prefix")

In [ ]:
# nglview: different representation styles
try:
    import nglview as nv
    
    view2 = nv.show_biopython(structure)
    view2.clear_representations()
    
    # Surface representation colored by hydrophobicity
    view2.add_surface(opacity=0.5, color='hydrophobicity')
    view2.add_cartoon(color='chainid', opacity=0.8)
    view2.background = 'white'
    
    print("Surface + cartoon representation:")
    display(view2)

except ImportError:
    print("nglview not available.")

### PyMOL Commands Reference

PyMOL is the most widely used molecular visualization program. Common commands:

```python
# Loading structures
fetch 1CRN            # Download from PDB
load myfile.pdb       # Load local file

# Representations
show cartoon          # Cartoon (helices, sheets, loops)
show sticks           # Stick model (bonds)
show surface          # Molecular surface
show spheres          # Space-filling (CPK)
show ribbon           # Ribbon trace

# Coloring
color red, chain A
color blue, ss h       # Color helices blue
color yellow, ss s     # Color sheets yellow
spectrum b             # Color by B-factor
spectrum count         # Rainbow from N to C terminus

# Selections
select active_site, resi 50+52+80+120
select helices, ss h
select ligand, hetatm and not resn HOH

# Analysis
distance d1, /1CRN/A/10/CA, /1CRN/A/20/CA    # Measure distance
angle a1, atom1, atom2, atom3                   # Measure angle
align mobile, target                            # Structural alignment
super mobile, target                            # Structure superposition
rms_cur mobile, target                          # RMSD of current position

# Surfaces
show surface
set surface_color, white
set transparency, 0.5

# Output
png output.png, dpi=300, ray=1    # High-quality image
save session.pse                   # Save session
```

---

## 10. Modern Structure Prediction: AlphaFold

### The Protein Folding Problem

Predicting a protein's 3D structure from its amino acid sequence has been one of the grand challenges in biology for over 50 years. In 2020, **AlphaFold2** (DeepMind) effectively solved this problem for single-domain proteins.

### How AlphaFold Works (Conceptual Overview)

```
Input: Amino acid sequence
         |
         v
  +---------------------+
  | Multiple Sequence    |   Search UniRef, BFD, etc.
  | Alignment (MSA)      |   for homologous sequences
  +---------------------+
         |             |
         v             v
  +-----------+  +-------------+
  | MSA       |  | Pair        |   Evolutionary covariance
  | embedding |  | embeddings  |   between residue pairs
  +-----------+  +-------------+
         \          /
          v        v
  +---------------------+
  | Evoformer (48 blocks)|   Attention mechanism
  | - MSA attention      |   extracts spatial
  | - Pair attention     |   relationships
  +---------------------+
         |
         v
  +---------------------+
  | Structure module     |   Predicts 3D coordinates
  | (8 iterations)       |   for each residue
  +---------------------+
         |
         v
  Output: 3D structure + pLDDT confidence scores
```

### pLDDT Confidence Score

| pLDDT | Confidence | Interpretation |
|-------|-----------|----------------|
| > 90 | Very high | Backbone and side-chains reliable |
| 70-90 | High | Backbone reliable |
| 50-70 | Low | Caution -- may be disordered |
| < 50 | Very low | Likely disordered or unstructured |

### Accessing AlphaFold Predictions

The **AlphaFold Protein Structure Database** (https://alphafold.ebi.ac.uk/) provides pre-computed predictions for over 200 million proteins. You can:

1. Search by UniProt accession
2. Download predicted structures as PDB or mmCIF files
3. The B-factor column in AlphaFold PDB files contains **pLDDT** confidence scores

In [ ]:
# Fetching an AlphaFold prediction
import urllib.request
import os

def fetch_alphafold_structure(uniprot_id, output_dir='pdb_files'):
    """
    Download an AlphaFold predicted structure from the EBI database.
    
    Parameters:
        uniprot_id: UniProt accession (e.g., 'P0DTD1' for SARS-CoV-2 nsp1)
        output_dir: Directory to save the file
    """
    os.makedirs(output_dir, exist_ok=True)
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    output_file = os.path.join(output_dir, f"AF-{uniprot_id}.pdb")
    
    try:
        urllib.request.urlretrieve(url, output_file)
        print(f"Downloaded AlphaFold structure for {uniprot_id}")
        return output_file
    except Exception as e:
        print(f"Could not download: {e}")
        return None


# Example: fetch AlphaFold prediction for crambin (UniProt: P01542)
af_file = fetch_alphafold_structure('P01542')

if af_file and os.path.exists(af_file):
    af_structure = PDBParser(QUIET=True).get_structure('alphafold', af_file)
    
    # In AlphaFold PDB files, B-factor = pLDDT confidence
    plddt_values = []
    for residue in af_structure[0].get_residues():
        if residue.id[0] == ' ' and 'CA' in residue:
            plddt = residue['CA'].get_bfactor()
            plddt_values.append(plddt)
    
    plddt_arr = np.array(plddt_values)
    print(f"\npLDDT statistics:")
    print(f"  Mean:    {plddt_arr.mean():.1f}")
    print(f"  Median:  {np.median(plddt_arr):.1f}")
    print(f"  Min:     {plddt_arr.min():.1f}")
    print(f"  Max:     {plddt_arr.max():.1f}")
    print(f"  High confidence (>70): {np.sum(plddt_arr > 70)}/{len(plddt_arr)} residues")

In [ ]:
# Compare experimental vs AlphaFold structure
if af_file and os.path.exists(af_file):
    # Extract CA coordinates from both structures
    exp_ca = []
    for res in structure[0]['A']:
        if res.id[0] == ' ' and 'CA' in res:
            exp_ca.append(res['CA'].get_vector().get_array())
    
    af_ca = []
    for res in af_structure[0].get_residues():
        if res.id[0] == ' ' and 'CA' in res:
            af_ca.append(res['CA'].get_vector().get_array())
    
    # Align on the shorter length
    n = min(len(exp_ca), len(af_ca))
    exp_arr = np.array(exp_ca[:n])
    af_arr = np.array(af_ca[:n])
    
    # Superpose
    aligned_af, rmsd_val, _ = kabsch_superpose(af_arr, exp_arr)
    tm = tm_score(aligned_af, exp_arr)
    
    print(f"Experimental vs AlphaFold comparison:")
    print(f"  Residues aligned: {n}")
    print(f"  RMSD:     {rmsd_val:.3f} A")
    print(f"  TM-score: {tm:.4f}")
    
    # Per-residue distance after alignment
    per_res_dist = np.sqrt(np.sum((aligned_af - exp_arr) ** 2, axis=1))
    print(f"\n  Per-residue deviation:")
    print(f"    Mean: {per_res_dist.mean():.3f} A")
    print(f"    Max:  {per_res_dist.max():.3f} A (residue {np.argmax(per_res_dist) + 1})")
else:
    print("AlphaFold structure not available for comparison.")

---

## 11. B-factor Analysis

The B-factor (temperature factor, Debye-Waller factor) reflects atomic displacement/flexibility. It is useful for:

- Identifying flexible loops and rigid cores
- Assessing model quality (very high B-factors may indicate poor electron density)
- Comparing dynamics across structures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract B-factors for CA atoms
b_factors = []
res_numbers = []

for residue in structure[0]['A']:
    if residue.id[0] == ' ' and 'CA' in residue:
        b_factors.append(residue['CA'].get_bfactor())
        res_numbers.append(residue.id[1])

b_arr = np.array(b_factors)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['red' if b > b_arr.mean() + b_arr.std() else 'steelblue' for b in b_factors]
ax.bar(res_numbers, b_factors, color=colors, alpha=0.7)
ax.axhline(y=b_arr.mean(), color='black', linestyle='--', label=f'Mean: {b_arr.mean():.1f}')
ax.axhline(y=b_arr.mean() + b_arr.std(), color='red', linestyle=':', 
           label=f'Mean + 1 SD: {b_arr.mean() + b_arr.std():.1f}')
ax.set_xlabel('Residue Number')
ax.set_ylabel('B-factor (A^2)')
ax.set_title('B-factor Profile -- Crambin (1CRN)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"B-factor statistics:")
print(f"  Mean: {b_arr.mean():.1f} A^2")
print(f"  Std:  {b_arr.std():.1f} A^2")
print(f"  Min:  {b_arr.min():.1f} A^2 (residue {res_numbers[np.argmin(b_arr)]})")
print(f"  Max:  {b_arr.max():.1f} A^2 (residue {res_numbers[np.argmax(b_arr)]})")

---

## 12. Exercises

### Exercise 1: Parse PDB and Extract Statistics

Download PDB entry **4HHB** (hemoglobin) and report:
- Number of chains and residues per chain
- Number of HETATM groups (identify the heme)
- Mean B-factor per chain

In [ ]:
# Exercise 1: Your solution here
# Hint:
# pdbl = PDBList()
# hemo_file = pdbl.retrieve_pdb_file('4HHB', pdir='pdb_files', file_format='pdb')
# hemo = PDBParser(QUIET=True).get_structure('hemoglobin', hemo_file)
# ...


### Exercise 2: Calculate Center of Mass

Write a function that calculates the center of mass of a protein chain, weighting each atom by its atomic mass. Compare this to the geometric center (unweighted average of coordinates).

Atomic masses: C=12.011, N=14.007, O=15.999, S=32.065, H=1.008

In [ ]:
# Exercise 2: Your solution here
ATOMIC_MASS = {'C': 12.011, 'N': 14.007, 'O': 15.999, 'S': 32.065, 'H': 1.008, 'SE': 78.96}

def center_of_mass(chain):
    """Calculate the mass-weighted center of a chain."""
    # Your code here
    pass

def geometric_center(chain):
    """Calculate the unweighted geometric center of a chain."""
    # Your code here
    pass

# Compare for crambin chain A
# com = center_of_mass(structure[0]['A'])
# gc = geometric_center(structure[0]['A'])
# print(f"Center of mass:     ({com[0]:.3f}, {com[1]:.3f}, {com[2]:.3f})")
# print(f"Geometric center:   ({gc[0]:.3f}, {gc[1]:.3f}, {gc[2]:.3f})")
# print(f"Difference: {distance(com, gc):.3f} A")

### Exercise 3: Find Active Site Residues

Download PDB entry **1HEW** (lysozyme with bound inhibitor NAG). Find all protein residues within 5 Angstroms of any ligand atom. These are candidate active site residues.

In [ ]:
# Exercise 3: Your solution here
# Hint:
# 1. Load 1HEW
# 2. Identify HETATM residues that are not water (res_name != 'HOH')
# 3. For each protein residue, check if any of its atoms is within 5 A
#    of any ligand atom
# 4. Report the list of nearby residues


### Exercise 4: Ramachandran Validation

Calculate phi/psi angles for PDB **2RNM** (a structure solved at lower resolution). Generate a Ramachandran plot and identify any outlier residues that fall outside the allowed regions. Outliers may indicate errors in the model.

In [ ]:
# Exercise 4: Your solution here
# Hint:
# 1. Download and parse 2RNM
# 2. Use calculate_phi_psi_biopython() from Section 6
# 3. Plot the Ramachandran diagram
# 4. Flag residues where phi > 0 (for non-glycine)
#    or that fall far from the expected alpha/beta regions


### Exercise 5: Structural Comparison of Homologs

Compare two globin structures:
1. Download **1MBO** (myoglobin) and **4HHB** (hemoglobin, chain A)
2. Extract CA atoms
3. Since the sequences differ in length, align only the first N matching residues
4. Superpose and calculate RMSD and TM-score
5. Plot per-residue distances after superposition

In [ ]:
# Exercise 5: Your solution here
# Hint:
# 1. pdbl.retrieve_pdb_file('1MBO', ...)
# 2. pdbl.retrieve_pdb_file('4HHB', ...)
# 3. Extract CA coords for myoglobin chain A and hemoglobin chain A
# 4. n = min(len(myo_ca), len(hemo_ca))
# 5. aligned, rmsd, R = kabsch_superpose(myo_ca[:n], hemo_ca[:n])
# 6. Plot per-residue distances


### Exercise 6: Disulfide Bond Detection

Crambin (1CRN) is stabilized by three disulfide bonds. Write code to automatically detect disulfide bonds by finding pairs of Cys SG atoms closer than 2.5 Angstroms.

In [ ]:
# Exercise 6: Your solution here
# Hint:
# 1. Find all CYS residues in chain A
# 2. Get the SG atom from each CYS
# 3. Check all pairs of SG atoms
# 4. If distance < 2.5 A, it's a disulfide bond
#
# cys_residues = [r for r in structure[0]['A'] if r.get_resname() == 'CYS']


---

## Summary

| Topic | Key Concepts |
|-------|-------------|
| Protein structure levels | Primary (sequence), secondary (helix/sheet), tertiary (3D fold), quaternary (multimer) |
| PDB file format | Fixed-width columns; ATOM records contain coordinates, B-factors, occupancy |
| BioPython Bio.PDB | SMCRA hierarchy: Structure > Model > Chain > Residue > Atom |
| Geometric measurements | Distance, angle, dihedral; RMSD for structural comparison |
| DSSP | Gold-standard secondary structure assignment from 3D coordinates |
| Ramachandran plot | phi/psi backbone angles; reveals allowed conformations and model errors |
| Structural alignment | Kabsch algorithm for superposition; TM-score for length-normalized comparison |
| Molecular surfaces | VdW, solvent-accessible (SASA), solvent-excluded; APBS for electrostatics |
| Visualization | nglview (Jupyter), PyMOL (standalone) |
| AlphaFold | Deep learning structure prediction; pLDDT confidence; AlphaFold DB |

---

## Resources

- [RCSB PDB](https://www.rcsb.org/) -- Protein Data Bank
- [PDB File Format v3.3](https://www.wwpdb.org/documentation/file-format)
- [BioPython Structural Bioinformatics FAQ](https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ)
- [DSSP](https://swift.cmbi.umcn.nl/gv/dssp/) -- Secondary structure assignment
- [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/)
- [PyMOL](https://pymol.org/) -- Molecular visualization
- [TM-align Server](https://zhanggroup.org/TM-align/) -- Structural alignment
- [MolProbity](http://molprobity.biochem.duke.edu/) -- Structure validation
- [APBS](https://www.poissonboltzmann.org/) -- Electrostatic calculations

---

[← Previous: Phylogenetics: Reconstructing Evolutionary History](../../Tier_2_Core_Bioinformatics/06_Phylogenetics/01_phylogenetics.ipynb) | [Next: Nucleic Acid Structure →](../../Tier_2_Core_Bioinformatics/08_Nucleic_Acid_Structure/01_nucleic_acid_structure.ipynb)